
### Libraries and Plot Settings

<div style="text-align: justify; text-justify: inter-word; font-size: 18px;">


We use NumPy for numerical calculations and Matplotlib for plotting
and animating the temperature distribution. Before starting the
simulation, we define a few plotting settings using Matplotlib's
<code>rcParams</code>, which controls the default appearance and
behavior of figures.



The default Matplotlib style is selected, and the figure size is set
to 5 × 4 inches. We also set <code>animation.html</code> to
<code>jshtml</code>, allowing the animation to be displayed directly
inside the Jupyter Notebook with interactive playback controls.
These settings only affect visualization and do not change the
numerical calculations.
</div>


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

plt.style.use("default")
plt.rcParams["figure.figsize"] = [5, 4]  # [width_inches, height_inches]
plt.rcParams["animation.html"] = "jshtml"

### Set up physical and numerical parameters

<div style="text-align: justify; text-justify: inter-word; font-size: 18px;">

The one-dimensional physical domain has a length of $L=1$ and is divided into $100$ equally spaced cells $(\Delta x=0.01)$. The computational nodes are located at the centers of the cells, with the first and last nodes positioned at $x=0.005$ and $x=0.995$, respectively. Therefore, no computational nodes lie directly on the physical boundaries $(x=0,\ x=L)$, as expected for a cell-centered FVM discretization.

The simulation consists of $50{,}000$ time steps, of which only $501$ snapshots are saved for the animation. Finally, the temperature array contains $102$ elements to accommodate the $100$ physical cells and two additional ghost cells used to impose the boundary conditions.
</div>

In [ ]:
L = 1.0
T = 0.5
dx = 0.01
dt = 1e-5
a = 1.0 # Thermal diffucivity
r = a * dt / dx**2

if r > 0.5: raise ValueError("Unstable problem: FTCS requires r <= 0.5")    # Stability conditon

# Cell generation 
nx = round(L / dx)
nt = round(T / dt)

x = (np.arange(nx)+0.5)*dx

u_old = np.empty(nx + 2)
u_new = np.empty(nx + 2)

# Save one profile every 100 time steps 
save_every = 100
n_save = nt // save_every + 1
u_saved = np.empty((n_save, nx))
t_saved = np.empty(n_save) 

### Example 1: Neumann Boundary Conditions

In [7]:
def Neumann(u_old,u_new,u_saved,t_saved):
    
    u_old[1:-1] = np.cos(np.pi*x)   # Initial condition

    # Save initial condition
    u_saved[0] = u_old[1:-1]
    t_saved[0] = 0.0
    k = 1

    for n in range(1,nt+1):
        u_old[0] = u_old[1]
        u_old[-1] = u_old[-2]
        u_new[1:-1] = r*u_old[2:] + (1 - 2*r)*u_old[1:-1] + r*u_old[:-2]
        
        u_old, u_new = u_new, u_old

        if n % save_every == 0:
            u_saved[k] = u_old[1:-1]
            t_saved[k] = n*dt
            k += 1
    return u_saved,t_saved
    
u_1,t_1 = Neumann(u_old,u_new,u_saved,t_saved)
u_1 = u_1.copy()
t_1 = t_1.copy()

### Example 1: Results, Animation and Discussion

In [ ]:
fig1, ax1 = plt.subplots()
line1, = ax1.plot(x, u_1[0])

ax1.set_xlim([0, L])
ax1.set_ylim([-1,1])
ax1.set_xlabel("x")
ax1.set_ylabel("Temperature")
ax1.grid(True,alpha=0.4)
title1 = ax1.set_title("t = 0.0000 s")

def update1(frame):

    line1.set_ydata(u_1[frame])
    title1.set_text(f"t = {t_1[frame]:.4f} s")
    return line1, title1

ani1 = FuncAnimation(fig1, update1,frames=len(u_1),interval=30,blit=False,repeat=False)
#ani1.save("neumann_animation.gif", writer="pillow", fps=20, dpi=100)
plt.close(fig1)
ani1

<div style="text-align: justify; text-justify: inter-word; font-size: 18px;">

Example 1 represents an initially compatible case, where the prescribed temperature gradients at both boundaries agree with those of the initial temperature distribution. As the simulation progresses, the temperature evolves smoothly according to the numerical scheme and gradually approaches zero, without any sudden changes, as expected from the analytical solution.

The previous code visualizes the results by plotting the temperature values against the computational coordinates $(x,u)$ and setting the horizontal axis limits to the physical domain $[0,L]$. However, as will be discussed in more detail in the next subsection, this approach does not include the temperatures at the physical boundaries. In a cell-centered FVM grid, the first and last computational nodes are located at $x=0.005$ and $x=0.995$, respectively, rather than at the boundaries themselves. This distinction is illustrated in the following figure.

![Neumann temperature distribution](https://raw.githubusercontent.com/ChrisPhys97/Heat_Equation/refs/heads/main/Compatibility%20and%20Ghost%20points/output.png)

A closer look at the figure reveals a very small gap near the right boundary, caused by the absence of a computational node at $x=L$. The corresponding gap at the left boundary is not immediately visible. However, by zooming into this region, we can clearly observe that the plotted temperature profile begins at the first cell center rather than at the physical boundary.

</div>

### Example 2: Time-dependent Dirichlet Boundary Conditions

In [ ]:
# --- Ramp Temperature Function ----

def g(t,Tr=0.1): return 0.5*min(t/Tr,1.0)

# ----------------------------------

def Dirichlet(u_old,u_new,u_saved,t_saved,g):   

    u_old[1:-1] = np.sin(np.pi*x)   

    # Save initial condition
    u_saved[0] = u_old[1:-1]
    t_saved[0] = 0.0
    k = 1

    for n in range(1, nt + 1):

        # Evaluate the boundary conditions at the previous time level    
        tn = (n-1)*dt

        # Update ghost-cell temperatures using the Dirichlet boundary conditions
        u_old[0] = 2*g(tn) - u_old[1]
        u_old[-1] = 2*g(tn) - u_old[-2]

        # Advance the solution
        u_new[1:-1] = r*u_old[2:] + (1 - 2*r)*u_old[1:-1] + r*u_old[:-2]

        u_old, u_new = u_new, u_old

        if n % save_every == 0:
            u_saved[k] = u_old[1:-1]
            t_saved[k] = n*dt
            k += 1
    return u_saved,t_saved

u_2,t_2 = Dirichlet(u_old,u_new,u_saved,t_saved,g)
u_2 = u_2.copy()
t_2 = t_2.copy()

## Example 2: 

### 1. Discussion

<div style="text-align: justify; text-justify: inter-word; font-size: 18px;">

Example 2 considers Dirichlet boundary conditions, where the temperature at both boundaries is prescribed. Initially, the temperature distribution is given by $u(x,0)=\sin(\pi x)$, which implies that the temperature at both boundaries is zero. If the boundary temperature is set immediately to $0.5$ for $t>0$ (starting from the first time step, $n=1$), the imposed boundary condition can be represented by a step function:

$$
g(t)=
\begin{cases}
0, & t=0,\\[4pt]
0.5, & t>0.
\end{cases}
$$

This represents a sudden temperature increase at both boundaries. Although the heat equation still admits a solution, the prescribed initial and boundary conditions are incompatible at the initial space-time corners $(0,0)$ and $(L,0)$. The discontinuity can be demonstrated by taking the right-hand limit:

$$
\lim_{t\to0^+}g(t)=0.5\neq g(0)=0.
$$

Therefore, the temperature is discontinuous at these two corners, even though the solution remains smooth in the interior of the domain for positive times. This discontinuity is a consequence of the prescribed boundary conditions rather than a numerical error introduced by the finite volume method.

To avoid this discontinuity, a ramp function is introduced to gradually increase the boundary temperature from $0$ to $0.5$, while simultaneously satisfying **zero-order compatibility**. This condition requires that the initial and boundary temperatures agree at $t=0$:

$$
u(0,0)=u(L,0)=g(0)=0.
$$

The ramp function is defined as

$$
g(t)=
\begin{cases}
0.5\dfrac{t}{T_r}, & 0\leq t<T_r,\\[6pt]
0.5, & t\geq T_r,
\end{cases}
$$

where $T_r$ is the transition time. Since $g(0)=0$ and the ramp function is continuous, the temperature remains continuous throughout the space-time domain, including the initial boundary corners.

Although the ramp function removes the discontinuity in temperature, it does not satisly the stronger compatibility condition required for certain higher-order derivatives to remain continous at the initial corners. For a sufficiently smooth solution we would need

$$g'(0) = \alpha u_{0}''(0)$$

For the initial condition 

$$u_{0}(x) = \sin(πx)$$

we have,

$$u_{0}''(0) = - \pi^2 \sin(0) = 0$$

but the ramp function gives

$$g'(0^+) =\dfrac{0.5}{T_r}$$

Thus,

$$g'(0^+) \neq a u_{0}''(0)$$

For the prescribed initial sine temperature profile, all even-order spatial derivatives vanish at both boundaries. Therefore, compatibility to every order requires a boundary temperature function whose derivatives also vanish at the initial time:

$$
\boxed{g^{(k)}(0)=0,\qquad k=0,1,2,\ldots}\qquad(1)
$$

A function that satisfies these compatibility conditions to every order is

$$
\boxed{
g(t)=
\begin{cases}
0, & t=0,\\[6pt]
0.5e^{-T_r/t}, & t>0,
\end{cases}
}\qquad(2)
$$

where $T_r>0$ is a characteristic time scale controlling how quickly the boundary temperature increases.

This function is infinitely differentiable and flat at the origin, meaning that it satisfies Eq. $(1)$. Therefore, it satisfies all orders of initial-boundary compatibility for the prescribed sine temperature profile. Furthermore, the boundary temperature approaches $0.5$ asymptotically:

$$
\boxed{
\lim_{t\to\infty}g(t)=0.5.
}
$$

Unlike the linear ramp function, the asymptotic function does not reach $0.5$ at a finite time, but approaches it continuously and smoothly.
</div>




### 2. Animation

<div style="text-align: justify; text-justify: inter-word; font-size: 18px;">

To display the temperature distribution across the entire domain,
we introduce two additional points in the plotting arrays.
Their positions correspond to the physical boundaries, while their
temperatures are taken directly from the prescribed Dirichlet
boundary conditions:

$$
u(0,t)=u(L,t)=g(t)
$$

The plotting arrays therefore contain N + 2 elements:

$$
x_{\mathrm{plot}}=[0,x_1,x_2,\ldots,x_N,L]
$$

$$
u_{\mathrm{plot}}=[g(t),u_1,u_2,\ldots,u_N,g(t)]
$$

These additional points are different from the ghost cells used in
the numerical algorithm. Ghost cells are located outside the physical
domain and allow us to apply the same CDS stencil to every physical
cell. The two plotting points, on the other hand, represent the actual
boundary locations.

Both plotting arrays are allocated once. For each animation frame,
the saved cell-center temperatures and the corresponding boundary
temperatures are assigned to the appropriate positions. In this way,
the animation shows the temperature distribution over the complete
domain without changing the numerical solution or introducing
additional control volumes.
</div>


In [ ]:
# Spatial coordinates, including the physical boundaries
x_plot = np.empty(len(x) + 2)
x_plot[0] = 0
x_plot[1:-1] = x
x_plot[-1] = L

# Temperature array for plotting (not ghost cells)
u_plot = np.empty(len(x) + 2)

# Initial temperature profile (ramp function)
u_plot[0] = g(t_2[0])
u_plot[1:-1] = u_2[0]
u_plot[-1] = g(t_2[0])

fig2, ax2 = plt.subplots()
line2, = ax2.plot(x_plot, u_plot)

ax2.set_xlim(0, L)
ax2.set_ylim(min(u_2.min(),0),max(u_2.max(),0.5))
ax2.set_xlabel("x")
ax2.set_ylabel("Temperature")
ax2.grid(True,alpha=0.4)
title2 = ax2.set_title("t = 0.0000 s")

def update2(frame):

    # Update the prescribed boundary temperatures
    u_plot[0] = g(t_2[frame])
    u_plot[-1] = g(t_2[frame])

    # Update the calculated cell-center temperatures
    u_plot[1:-1] = u_2[frame]

    line2.set_ydata(u_plot)
    title2.set_text(f"t = {t_1[frame]:.4f} s")

    return line2, title2

ani2 = FuncAnimation(fig2, update2,frames=len(u_1),interval=30,blit=True,repeat=False)
ani2.save("Dirichlet_animation.gif", writer="pillow", fps=20, dpi=100)
plt.close(fig2)
ani2